In [1]:
import torch
from torch import nn, optim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# Separar os ataques

In [2]:
path = '../Artigos/Deteccao_de_anomalia_em_fluxo_de_rede/Datasets/CICIDS2018/archive/'
filenames = ['02_14_2018','02_15_2018','02_16_2018','02_20_2018','02_21_2018','02_22_2018','02_23_2018','02_28_2018','03_01_2018','03_02_2018']
day4 = pd.read_csv(path + filenames[3] + '.csv') #Dia 4 muito grande

In [3]:
day4.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,172.31.69.25-94.231.103.172-22-45498-6,94.231.103.172,45498,172.31.69.25,22,6,20/02/2018 08:34:07,888751,11,11,...,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
1,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:33:22,112642816,3,0,...,0,0.0,0.0,0.0,0.0,56300000.0,7.071068,56300000.0,56300000.0,Benign
2,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:36:11,112642712,3,0,...,0,0.0,0.0,0.0,0.0,56300000.0,18.384776,56300000.0,56300000.0,Benign
3,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:39:00,112642648,3,0,...,0,0.0,0.0,0.0,0.0,56300000.0,5.656854,56300000.0,56300000.0,Benign
4,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:41:49,112642702,3,0,...,0,0.0,0.0,0.0,0.0,56300000.0,65.053824,56300000.0,56300000.0,Benign


In [4]:
day4 = day4.drop(columns=['Flow ID','Src IP','Src Port','Dst IP','Timestamp'])

day4.head()

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,6,888751,11,11,1249.0,1969.0,736.0,0.0,113.545455,...,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign
1,0,0,112642816,3,0,0.0,0.0,0.0,0.0,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,7.071068,56300000.0,56300000.0,Benign
2,0,0,112642712,3,0,0.0,0.0,0.0,0.0,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,18.384776,56300000.0,56300000.0,Benign
3,0,0,112642648,3,0,0.0,0.0,0.0,0.0,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,5.656854,56300000.0,56300000.0,Benign
4,0,0,112642702,3,0,0.0,0.0,0.0,0.0,0.000000,...,0,0.0,0.0,0.0,0.0,56300000.0,65.053824,56300000.0,56300000.0,Benign


In [5]:
day4['Label'].value_counts()

Label
Benign                    7372557
DDoS attacks-LOIC-HTTP     576191
Name: count, dtype: int64

In [6]:
DDoS_LOIC_HTTP = day4[day4['Label'] == 'DDoS attacks-LOIC-HTTP']
benign4 = day4[day4['Label'] == 'Benign']
day4 = []

In [7]:
DDoS_LOIC_HTTP.to_csv('attack_data/DDoS_LOIC_HTTP.csv', index = False)

In [10]:
benign4.to_csv('attack_data/benign4.csv', index = False)

In [12]:
day1 = pd.read_csv(path + filenames[0] + '.csv')
day2 = pd.read_csv(path + filenames[1] + '.csv')
day3 = pd.read_csv(path + filenames[2] + '.csv')
# day4 = pd.read_csv(path + filenames[3] + '.csv') #Dia 4 muito grande, usando o mini (benigno reduzido para balancear com maligno)
day5 = pd.read_csv(path + filenames[4] + '.csv')
day6 = pd.read_csv(path + filenames[5] + '.csv')
day7 = pd.read_csv(path + filenames[6] + '.csv')
day8 = pd.read_csv(path + filenames[7] + '.csv')
day9 = pd.read_csv(path + filenames[8] + '.csv')
day10 = pd.read_csv(path + filenames[9] + '.csv')

C:\Users\linco\AppData\Local\Temp\ipykernel_33772\1046280479.py:3: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  day3 = pd.read_csv(path + filenames[2] + '.csv')
C:\Users\linco\AppData\Local\Temp\ipykernel_33772\1046280479.py:8: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  day8 = pd.read_csv(path + filenames[7] + '.csv')
C:\Users\linco\AppData\Local\Temp\ipykernel_33772\1046280479.py:9: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,1

In [13]:
day1['Label'].value_counts()

Label
Benign            667626
FTP-BruteForce    193360
SSH-Bruteforce    187589
Name: count, dtype: int64

In [ ]:
benign1 = day1[day1['Label'] == 'Benign']
FTP_BruteForce = day1[day1['Label'] == 'FTP-BruteForce']
SSH_Bruteforce = day1[day1['Label'] == 'SSH-Bruteforce']

In [17]:
benign1.to_csv('attack_data/benign1.csv', index = False)

In [18]:
FTP_BruteForce.to_csv('attack_data/FTP_BruteForce.csv', index = False)


In [19]:
SSH_Bruteforce.to_csv('attack_data/SSH_Bruteforce.csv', index = False)


In [21]:
day2['Label'].value_counts()

Label
Benign                   996077
DoS attacks-GoldenEye     41508
DoS attacks-Slowloris     10990
Name: count, dtype: int64

In [22]:
benign2 = day2[day2['Label'] == 'Benign']
DoS_GoldenEye = day2[day2['Label'] == 'DoS attacks-GoldenEye']
DoS_Slowloris = day2[day2['Label'] == 'DoS attacks-Slowloris']

In [23]:
benign2.to_csv('attack_data/benign2.csv', index = False)
DoS_GoldenEye.to_csv('attack_data/DoS_GoldenEye.csv', index = False)
DoS_Slowloris.to_csv('attack_data/DoS_Slowloris.csv', index = False)

In [25]:
day3['Label'].value_counts()

Label
DoS attacks-Hulk            461912
Benign                      446772
DoS attacks-SlowHTTPTest    139890
Label                            1
Name: count, dtype: int64

In [26]:
benign3 = day3[day3['Label'] == 'Benign']
DoS_Hulk = day3[day3['Label'] == 'DoS attacks-Hulk']
DoS_SlowHTTPTest = day3[day3['Label'] == 'DoS attacks-SlowHTTPTest']

In [27]:
benign3.to_csv('attack_data/benign3.csv', index = False)
DoS_Hulk.to_csv('attack_data/DoS_Hulk.csv', index = False)
DoS_SlowHTTPTest.to_csv('attack_data/DoS_SlowHTTPTest.csv', index = False)

In [29]:
day5['Label'].value_counts()

Label
DDOS attack-HOIC        686012
Benign                  360833
DDOS attack-LOIC-UDP      1730
Name: count, dtype: int64

In [30]:
benign5 = day5[day5['Label'] == 'Benign']
DDOS_HOIC = day5[day5['Label'] == 'DDOS attack-HOIC']
DDOS_LOIC_UDP = day5[day5['Label'] == 'DDOS attack-LOIC-UDP']

In [31]:
benign5.to_csv('attack_data/benign5.csv', index = False)
DDOS_HOIC.to_csv('attack_data/DDOS_HOIC.csv', index = False)
DDOS_LOIC_UDP.to_csv('attack_data/DDOS_LOIC_UDP.csv', index = False)

In [33]:
day6['Label'].value_counts()

Label
Benign              1048213
Brute Force -Web        249
Brute Force -XSS         79
SQL Injection            34
Name: count, dtype: int64

In [34]:
benign6 = day6[day6['Label'] == 'Benign']
BruteForce_Web = day6[day6['Label'] == 'Brute Force -Web']
BruteForce_XSS = day6[day6['Label'] == 'Brute Force -XSS']
SQLInjection = day6[day6['Label'] == 'SQL Injection']

In [35]:
benign6.to_csv('attack_data/benign6.csv', index = False)
BruteForce_Web.to_csv('attack_data/BruteForce_Web.csv', index = False)
BruteForce_XSS.to_csv('attack_data/BruteForce_XSS.csv', index = False)
SQLInjection.to_csv('attack_data/SQLInjection.csv', index = False)

In [43]:
day7['Label'].value_counts()

Label
Benign              1048009
Brute Force -Web        362
Brute Force -XSS        151
SQL Injection            53
Name: count, dtype: int64

In [44]:
benign7 = day7[day7['Label'] == 'Benign']
BruteForce_Web = day7[day7['Label'] == 'Brute Force -Web']
BruteForce_XSS = day7[day7['Label'] == 'Brute Force -XSS']
SQLInjection = day7[day7['Label'] == 'SQL Injection']

In [45]:
benign7.to_csv('attack_data/benign7.csv', index = False)
BruteForce_Web.to_csv('attack_data/BruteForce_Web2.csv', index = False)
BruteForce_XSS.to_csv('attack_data/BruteForce_XSS2.csv', index = False)
SQLInjection.to_csv('attack_data/SQLInjection2.csv', index = False)

In [47]:
day8['Label'].value_counts()

Label
Benign           544200
Infilteration     68871
Label                33
Name: count, dtype: int64

In [48]:
benign8 = day8[day8['Label'] == 'Benign']
Infilteration = day8[day8['Label'] == 'Infilteration']

In [49]:
benign8.to_csv('attack_data/benign8.csv', index = False)
Infilteration.to_csv('attack_data/Infilteration1.csv', index = False)

In [50]:
day9['Label'].value_counts()

Label
Benign           238037
Infilteration     93063
Label                25
Name: count, dtype: int64

In [51]:
benign9 = day9[day9['Label'] == 'Benign']
Infilteration = day9[day9['Label'] == 'Infilteration']

In [52]:
benign9.to_csv('attack_data/benign9.csv', index = False)
Infilteration.to_csv('attack_data/Infilteration2.csv', index = False)

In [54]:
day10['Label'].value_counts()

Label
Benign    762384
Bot       286191
Name: count, dtype: int64

In [55]:
benign10 = day10[day10['Label'] == 'Benign']
Bot = day10[day10['Label'] == 'Bot']

In [56]:
benign10.to_csv('attack_data/benign10.csv', index = False)
Bot.to_csv('attack_data/Bot.csv', index = False)

# Juntar os benignos

In [2]:
total_benign = pd.read_csv('attack_data/benign1.csv')
total_benign = total_benign.drop(columns=['Timestamp'])
for i in range(2,11):
    b = pd.read_csv(f'attack_data/benign{i}.csv')
    if 'Timestamp' in b.columns:
        b = b.drop(columns=['Timestamp'])

    total_benign = pd.concat([total_benign,b], ignore_index=True)

In [3]:
total_benign.shape

(13484708, 79)

In [4]:
total_benign.to_csv('attack_data/benign.csv', index = False)

In [8]:
benign_6 = total_benign.sample(frac=0.06, random_state=123)

In [9]:
benign_6.shape

(809082, 79)

In [10]:
benign_6.to_csv('attack_data/benign_6pc.csv', index = False)

# Juntar os ataques em múltiplos dias

In [12]:
atks = ['BruteForce_Web', 'BruteForce_XSS', 'SQLInjection', 'Infilteration']

for a in atks:
    atk1 = pd.read_csv(f'attack_data/{a}1.csv')
    atk2 = pd.read_csv(f'attack_data/{a}2.csv')
    atk = pd.concat([atk1,atk2], ignore_index=True)
    atk = atk.drop(columns=['Timestamp'])
    atk.to_csv(f'attack_data/{a}.csv', index = False)
    print(f'{a} {atk.shape}')


BruteForce_Web (611, 79)
BruteForce_XSS (230, 79)
SQLInjection (87, 79)
Infilteration (161934, 79)


# Dropar os timestamps faltantes

In [4]:
import os

files = os.listdir('attack_data/')
files.remove('benign.csv')

for f in files:
    df = pd.read_csv(f'attack_data/{f}')
    if 'Timestamp' in df.columns:
        df = df.drop(columns=['Timestamp'])
        print(f'{f} {df.shape}')
        df.to_csv(f'attack_data/{f}', index = False)


Bot.csv (286191, 79)
DDOS_HOIC.csv (686012, 79)
DDOS_LOIC_UDP.csv (1730, 79)
DoS_GoldenEye.csv (41508, 79)
DoS_Hulk.csv (461912, 79)
DoS_SlowHTTPTest.csv (139890, 79)
DoS_Slowloris.csv (10990, 79)
FTP_BruteForce.csv (193360, 79)
SSH_Bruteforce.csv (187589, 79)


# Juntar ataques e benign_6%

In [5]:
import os
files = os.listdir('attack_data/')
files.remove('benign.csv')
all_classes_raw = pd.read_csv(f'attack_data/{files[0]}')
for f in files[1:]:
    df = pd.read_csv(f'attack_data/{f}')
    all_classes_raw = pd.concat([all_classes_raw, df], ignore_index=True)

all_classes_raw.to_csv(f'attack_data/all_classes_raw.csv', index = False)

# OHE do Dst Port e Protocol

In [2]:
acr = pd.read_csv('attack_data/all_classes_raw.csv')

acr['Dst Port'].value_counts()

Dst Port
80       1886476
21        333344
53        291243
8080      281701
22        189081
          ...   
12577          1
2828           1
42149          1
35563          1
40933          1
Name: count, Length: 37596, dtype: int64

In [3]:
acr['Dst Port'].value_counts()[:10]

Dst Port
80      1886476
21       333344
53       291243
8080     281701
22       189081
443      149952
3389     122314
445       47803
0         16579
135        3317
Name: count, dtype: int64

In [4]:
acr['Dst Port'].value_counts()[10:].sum()

np.int64(235507)

In [20]:
hf = []
mf = []
lf = []

for p, value in acr['Dst Port'].value_counts().items():
    if value > 10000:
        hf.append(p)
    elif value > 1000:
        mf.append(p)
    else:
        lf.append(p)


In [4]:
top9_ports = list(acr['Dst Port'].value_counts()[:9].keys())

for port in top9_ports:
    acr['ohe_port_' + str(port)] = (acr['Dst Port'] == port).astype(int)

acr['ohe_port_other'] = (~acr['Dst Port'].isin(top9_ports)).astype(int)

# acr['ohe_port_hf'] = (acr['Dst Port'].isin(hf)).astype(int)
# acr['ohe_port_mf'] = (acr['Dst Port'].isin(mf)).astype(int)
# acr['ohe_port_lf'] = (acr['Dst Port'].isin(lf)).astype(int)

In [5]:
acr.head()

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,ohe_port_80,ohe_port_21,ohe_port_53,ohe_port_8080,ohe_port_22,ohe_port_443,ohe_port_3389,ohe_port_445,ohe_port_0,ohe_port_other
0,80,6,552910,5,3,515.0,298.0,515.0,0.0,103.0,...,1,0,0,0,0,0,0,0,0,0
1,53,17,327353,2,2,82.0,146.0,41.0,41.0,41.0,...,0,0,1,0,0,0,0,0,0,0
2,51890,6,284,2,1,38.0,0.0,38.0,0.0,19.0,...,0,0,0,0,0,0,0,0,0,1
3,53,17,436,1,1,32.0,48.0,32.0,32.0,32.0,...,0,0,1,0,0,0,0,0,0,0
4,3389,6,1717585,8,7,1032.0,1429.0,565.0,0.0,129.0,...,0,0,0,0,0,0,1,0,0,0


In [6]:
top_protocols = list(acr['Protocol'].value_counts().keys())

for prot in top_protocols:
    acr['ohe_prot_' + str(prot)] = (acr['Protocol'] == prot).astype(int)

In [7]:
acr.head()

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,ohe_port_8080,ohe_port_22,ohe_port_443,ohe_port_3389,ohe_port_445,ohe_port_0,ohe_port_other,ohe_prot_6,ohe_prot_17,ohe_prot_0
0,80,6,552910,5,3,515.0,298.0,515.0,0.0,103.0,...,0,0,0,0,0,0,0,1,0,0
1,53,17,327353,2,2,82.0,146.0,41.0,41.0,41.0,...,0,0,0,0,0,0,0,0,1,0
2,51890,6,284,2,1,38.0,0.0,38.0,0.0,19.0,...,0,0,0,0,0,0,1,1,0,0
3,53,17,436,1,1,32.0,48.0,32.0,32.0,32.0,...,0,0,0,0,0,0,0,0,1,0
4,3389,6,1717585,8,7,1032.0,1429.0,565.0,0.0,129.0,...,0,0,0,1,0,0,0,1,0,0


In [8]:
acr = acr.drop(columns=['Dst Port','Protocol'])

In [9]:
acr.head()

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,ohe_port_8080,ohe_port_22,ohe_port_443,ohe_port_3389,ohe_port_445,ohe_port_0,ohe_port_other,ohe_prot_6,ohe_prot_17,ohe_prot_0
0,552910,5,3,515.0,298.0,515.0,0.0,103.0,230.315002,298.0,...,0,0,0,0,0,0,0,1,0,0
1,327353,2,2,82.0,146.0,41.0,41.0,41.0,0.000000,73.0,...,0,0,0,0,0,0,0,0,1,0
2,284,2,1,38.0,0.0,38.0,0.0,19.0,26.870058,0.0,...,0,0,0,0,0,0,1,1,0,0
3,436,1,1,32.0,48.0,32.0,32.0,32.0,0.000000,48.0,...,0,0,0,0,0,0,0,0,1,0
4,1717585,8,7,1032.0,1429.0,565.0,0.0,129.0,190.919579,1149.0,...,0,0,0,1,0,0,0,1,0,0


# Juntando classes de ataques (14 -> 7)

In [10]:
# TRAFFIC_LABEL = {'Benign': 0,
#                  'FTP-BruteForce': 1, 'SSH-Bruteforce': 1,
#                  'DoS attacks-GoldenEye': 2, 'DoS attacks-Slowloris': 2,
#                  'DoS attacks-SlowHTTPTest': 2, 'DoS attacks-Hulk': 2,
#                  'Infilteration': 3,
#                  'DDoS attacks-LOIC-HTTP': 4, 'DDOS attack-LOIC-UDP': 4, 'DDOS attack-HOIC': 4,
#                  'Brute Force -Web': 5,
#                  'Brute Force -XSS': 5,
#                  'Bot': 6,
#                  'SQL Injection': 7
#                  }

acr['Label'] = acr['Label'].replace('FTP-BruteForce','BruteForce')
acr['Label'] = acr['Label'].replace('SSH-Bruteforce','BruteForce')
acr['Label'] = acr['Label'].replace('DoS attacks-GoldenEye','DoS')
acr['Label'] = acr['Label'].replace('DoS attacks-Slowloris','DoS')
acr['Label'] = acr['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
acr['Label'] = acr['Label'].replace('DoS attacks-Hulk','DoS')
acr['Label'] = acr['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
acr['Label'] = acr['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
acr['Label'] = acr['Label'].replace('DDOS attack-HOIC','DDoS')
acr['Label'] = acr['Label'].replace('Brute Force -Web','Web')
acr['Label'] = acr['Label'].replace('Brute Force -XSS','Web')
acr['Label'] = acr['Label'].replace('SQL Injection','Web')

# Normalização dos dados

In [13]:
num_cols = list(acr.columns)
num_cols.remove('Label')
num_ohe = sum(1 for element in num_cols if element[:3] == 'ohe')

In [14]:
acr = acr.replace(np.inf,np.nan)
acr = acr.dropna()

In [15]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
norm = scaler.fit_transform(acr.to_numpy()[:, :-(num_ohe + 1)])

In [18]:
acn = pd.DataFrame(norm, columns=num_cols[:-num_ohe])
acn

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,0.999814,0.000013,0.000024,0.000038,1.905853e-06,0.025688,0.000000,0.017072,0.051853,0.004573,...,0.000003,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.999814,0.000003,0.000016,0.000006,9.337401e-07,0.002045,0.028082,0.006796,0.000000,0.001120,...,0.000003,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.999813,0.000003,0.000008,0.000003,0.000000e+00,0.001895,0.000000,0.003149,0.006049,0.000000,...,0.000000,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.999813,0.000000,0.000008,0.000002,3.069830e-07,0.001596,0.021918,0.005304,0.000000,0.000737,...,0.000000,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.999816,0.000023,0.000057,0.000077,9.139141e-06,0.028182,0.000000,0.021381,0.042983,0.017634,...,0.000016,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550283,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3550284,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3550285,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3550286,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
for c in num_cols[-num_ohe:]:
    acn[c] = acr[c].values

acn

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,ohe_port_8080,ohe_port_22,ohe_port_443,ohe_port_3389,ohe_port_445,ohe_port_0,ohe_port_other,ohe_prot_6,ohe_prot_17,ohe_prot_0
0,0.999814,0.000013,0.000024,0.000038,1.905853e-06,0.025688,0.000000,0.017072,0.051853,0.004573,...,0,0,0,0,0,0,0,1,0,0
1,0.999814,0.000003,0.000016,0.000006,9.337401e-07,0.002045,0.028082,0.006796,0.000000,0.001120,...,0,0,0,0,0,0,0,0,1,0
2,0.999813,0.000003,0.000008,0.000003,0.000000e+00,0.001895,0.000000,0.003149,0.006049,0.000000,...,0,0,0,0,0,0,1,1,0,0
3,0.999813,0.000000,0.000008,0.000002,3.069830e-07,0.001596,0.021918,0.005304,0.000000,0.000737,...,0,0,0,0,0,0,0,0,1,0
4,0.999816,0.000023,0.000057,0.000077,9.139141e-06,0.028182,0.000000,0.021381,0.042983,0.017634,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550283,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,1,0,0,0,0,0,1,0,0
3550284,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,1,0,0,0,0,0,1,0,0
3550285,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,1,0,0,0,0,0,1,0,0
3550286,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,1,0,0,0,0,0,1,0,0


In [20]:
acn['Label'] = acr['Label'].values
acn

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,...,ohe_port_22,ohe_port_443,ohe_port_3389,ohe_port_445,ohe_port_0,ohe_port_other,ohe_prot_6,ohe_prot_17,ohe_prot_0,Label
0,0.999814,0.000013,0.000024,0.000038,1.905853e-06,0.025688,0.000000,0.017072,0.051853,0.004573,...,0,0,0,0,0,0,1,0,0,Benign
1,0.999814,0.000003,0.000016,0.000006,9.337401e-07,0.002045,0.028082,0.006796,0.000000,0.001120,...,0,0,0,0,0,0,0,1,0,Benign
2,0.999813,0.000003,0.000008,0.000003,0.000000e+00,0.001895,0.000000,0.003149,0.006049,0.000000,...,0,0,0,0,0,1,1,0,0,Benign
3,0.999813,0.000000,0.000008,0.000002,3.069830e-07,0.001596,0.021918,0.005304,0.000000,0.000737,...,0,0,0,0,0,0,0,1,0,Benign
4,0.999816,0.000023,0.000057,0.000077,9.139141e-06,0.028182,0.000000,0.021381,0.042983,0.017634,...,0,0,1,0,0,0,1,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550283,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,0,0,0,0,0,1,0,0,BruteForce
3550284,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,0,0,0,0,0,1,0,0,BruteForce
3550285,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,0,0,0,0,0,1,0,0,BruteForce
3550286,0.999813,0.000000,0.000008,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,0,0,0,0,0,1,0,0,BruteForce


In [21]:
acn['Label'].unique()

array(['Benign', 'Bot', 'Web', 'DDoS', 'DoS', 'BruteForce',
       'Infilteration'], dtype=object)

In [22]:
acn.to_csv(f'attack_data/all_classes_normalized_10_port_joined_attacks.csv', index = False)